In [ ]:
# One-cell Kaggle: enable GPU + Internet, then run. Full LOSO S01-S09.
import os, shutil, subprocess, sys
from pathlib import Path

# Kaggle injects an inline backend that is unavailable inside the isolated venv.
os.environ['MPLBACKEND'] = 'Agg'

BRANCH = 'feature/hada-full-mamba-lk-temporal'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/kaggle/working/tcformer-full-mamba-lk-bcic2a')
RESULT_ARCHIVE = Path('/kaggle/working/full_mamba_lk_bcic2a_loso_s01_s09_ep125')

def run(command, cwd=None, env=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', BRANCH], cwd=REPO_PATH)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)
uv = shutil.which('uv') or 'uv'
run([uv, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
python = REPO_PATH / '.venv/bin/python'
run([uv, 'pip', 'install', '--python', python, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([uv, 'pip', 'install', '--python', python, '-r', 'requirements.txt'], cwd=REPO_PATH)
run([python, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_lk_temporal.py', '-v'], cwd=REPO_PATH)
run(['nvidia-smi'])
run([python, '-c', "import torch; assert torch.cuda.is_available(), 'Enable a Kaggle GPU'; print('GPU:', torch.cuda.get_device_name(0))"])
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'MPLBACKEND': 'Agg', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})
print('===== FULL-MAMBA + LK | BCI IV-2a LOSO S01-S09 | RA=True | IM-TTA=5 | EPOCHS=125 =====', flush=True)
run([python, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', 'bcic2a', '--loso', '--gpu_id', '0'], cwd=REPO_PATH, env=environment)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Kaggle output:', archive, flush=True)
